[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/38.%20The%20Automation%20Investment%20Analysis%20Problem/P38-Tier-7_executed.ipynb)

---

# 38. The Automation Investment Analysis Problem
## Tier 7: The Human-AI Symbiotic Partnership (The Centaur Model)

### Goal and scope

This notebook presents an illustrative human-AI decision process for selecting a portfolio of port-automation investments. A simulated AI system samples candidate portfolios, scores them against a budget and stated priorities, and explains its recommendation. Human experts then review the recommendation, provide structured feedback, and contribute to implementation planning. The notebook also compares the resulting measures with illustrative individual-approach benchmarks and demonstrates a hypothetical learning trajectory.

The example is a teaching simulation, not an empirical study or a validated investment appraisal. The project values, historical success rates, expert assessments, collaboration benefits, and benchmark metrics are assumed for illustration. In particular, the simulated historical data do not represent verified observations from real terminals, and the benchmark comparison does not establish that collaboration is generally superior.

**Inputs:** five projects with costs, expected net present values (NPVs), implementation times, risk levels, and stakeholder impacts; four expert profiles; a budget; and a decision context.

**Outputs:** a sampled AI recommendation with reasoning and sensitivity summaries; a human-reviewed portfolio and implementation plan; illustrative comparison metrics; and visual and learning summaries.

**Units:** project costs and NPVs are expressed in millions of dollars (\$M); implementation times are in months; risk, confidence, agreement, and other scores are dimensionless values on a 0–1 scale unless stated otherwise.

### Modeling assumptions

- Project NPVs are additive; interactions, financing, taxes, discount-rate uncertainty, and dependencies between projects are not modeled.
- The scenario generator samples candidate project subsets and market conditions. Its best-scoring candidate is a best sampled candidate, not a guarantee of a global optimum.
- Market multipliers are illustrative scenario assumptions. Reported portfolio NPVs use the projects' base NPVs so that the AI, human, and collaborative benchmark values are comparable; market effects are shown separately in sensitivity analysis.
- Human priorities and feedback are generated from expert profiles using random variation. Stakeholder effects and implementation outcomes are qualitative or illustrative rather than measured forecasts.
- The learning curve is specified for demonstration. It is not estimated from observed collaboration cycles.

### Method and mathematical representation

Let \(P\) be the set of projects, and let \(S\subseteq P\) be a candidate portfolio. For project \(p\), let \(c_p\) be its cost and \(v_p\) its base expected NPV. A sampled scenario also has a market condition \(m\) with multiplier \(q_m\). Candidate portfolios that exceed the decision budget \(B\) are excluded. The scenario generator first creates a random subset and scenario budget; if the subset exceeds that scenario budget, it retains projects greedily in descending order of \(v_p/c_p\) until the budget is met.

For the resulting scenario, the market-adjusted NPV is \(q_m\sum_{p\in S}v_p\). The recommendation stage scores each scenario using the available strategic priorities. In this example, the implemented score is

$$\operatorname{Score}(S,m)=q_m\sum_{p\in S}v_p\,F(S),$$

where the multiplicative priority factor \(F(S)\) applies the financial-return, innovation, and risk-averse adjustments when those priority keys are present. The system chooses the highest-scoring scenario among the generated scenarios satisfying \(\sum_{p\in S}c_p\leq B\). It reports the selected portfolio's base NPV and base-NPV-to-cost ratio, while the market sensitivity analysis displays the effect of \(q_m\).

The code samples 10,000 scenarios. Sorting and scoring therefore depend on the number of sampled scenarios and the number of projects in each candidate; this is not exhaustive enumeration of all possible portfolios. Because sampling is random and no random seed is set, repeated runs may produce different recommendations and illustrative metrics.

### Collaboration process

The decision proceeds in five phases: problem framing, AI analysis, human review, collaborative refinement, and implementation planning. Expert priorities are aggregated using their strategic weights. Human feedback can remove an autonomous-crane project when labor concerns are raised and can add predictive maintenance only when the resulting portfolio remains within budget. The trust update moves each expert's trust score toward the measured portfolio-overlap agreement by a learning-rate fraction.

For AI portfolio \(A\) and refined portfolio \(R\), portfolio overlap is measured by the Jaccard similarity

$$J(A,R)=\frac{|A\cap R|}{|A\cup R|},$$

with a default value of 0.5 if both portfolios are empty. The reported agreement blends this overlap with mean pre-update trust: \(0.7J(A,R)+0.3\bar{t}\). For expert \(i\), trust is then updated as \(t_i' = \min(0.95,\max(0.1,t_i+\alpha(a-t_i)))\), where \(a\) is the agreement and \(\alpha\) is the learning rate. These are illustrative bookkeeping rules, not empirically validated behavioral models.

### What to look for

Check whether the selected portfolio respects the budget, whether the reasoning and sensitivity summaries match the model, and how human feedback changes the portfolio and agreement measure. Interpret the benchmark table and learning curves as consequences of their stated assumptions—not as evidence that one decision approach will outperform another in practice.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")
print("Libraries imported for the human-AI decision simulation.")

In [ ]:
@dataclass
class HumanExpert:
    """Profile of an expert participating in the decision process."""
    id: str
    name: str
    expertise: List[str]
    experience_years: int
    strategic_weight: float
    risk_tolerance: float
    stakeholder_focus: List[str]


@dataclass
class AIRecommendation:
    """Recommendation and supporting summaries from the simulated AI."""
    recommended_portfolio: List[str]
    expected_npv: float
    confidence_score: float
    reasoning_path: List[str]
    sensitivity_analysis: Dict[str, Dict[str, float]]
    alternative_solutions: List[Dict]
    risk_assessment: Dict[str, float]


@dataclass
class HumanInput:
    """Structured input generated for one expert."""
    expert_id: str
    strategic_priorities: Dict[str, float]
    stakeholder_concerns: Dict[str, str]
    risk_factors: List[str]
    implementation_considerations: List[str]
    qualitative_factors: Dict[str, float]


@dataclass
class CollaborativeDecision:
    """Portfolio, outcomes, and planning summaries from collaboration."""
    selected_portfolio: List[str]
    expected_outcomes: Dict[str, float]
    stakeholder_impact: Dict[str, str]
    implementation_plan: Dict
    confidence_level: float
    human_ai_agreement: float


@dataclass
class PartnershipMetrics:
    """Illustrative measures for a collaboration cycle."""
    decision_quality_score: float
    stakeholder_satisfaction: float
    implementation_success: float
    trust_level: float
    learning_rate: float


investment_projects = {
    "autonomous_cranes": {
        "name": "Autonomous Crane System", "cost": 35.0, "expected_npv": 45.0,
        "implementation_time": 18, "risk_level": 0.40,
        "stakeholder_impact": {"labor": "high", "operations": "high", "safety": "medium", "community": "low"}
    },
    "ai_yard_management": {
        "name": "AI-Powered Yard Management", "cost": 25.0, "expected_npv": 32.0,
        "implementation_time": 12, "risk_level": 0.25,
        "stakeholder_impact": {"labor": "medium", "operations": "high", "customers": "medium", "competition": "high"}
    },
    "predictive_maintenance": {
        "name": "Predictive Maintenance AI", "cost": 15.0, "expected_npv": 18.0,
        "implementation_time": 6, "risk_level": 0.15,
        "stakeholder_impact": {"labor": "low", "maintenance": "high", "operations": "medium", "suppliers": "medium"}
    },
    "digital_twin": {
        "name": "Integrated Digital Twin", "cost": 20.0, "expected_npv": 28.0,
        "implementation_time": 15, "risk_level": 0.30,
        "stakeholder_impact": {"engineering": "high", "operations": "medium", "customers": "low", "innovation": "high"}
    },
    "smart_gates": {
        "name": "Smart Gate Automation", "cost": 12.0, "expected_npv": 15.0,
        "implementation_time": 8, "risk_level": 0.20,
        "stakeholder_impact": {"truckers": "high", "operations": "medium", "community": "medium", "environment": "low"}
    }
}

human_experts = [
    HumanExpert("ceo", "Sarah Chen", ["strategic_planning", "market_positioning", "financial_analysis"], 15, 0.9, 0.6, ["investors", "board", "competition"]),
    HumanExpert("operations_director", "Michael Rodriguez", ["terminal_operations", "logistics", "process_optimization"], 20, 0.8, 0.4, ["labor", "operations", "safety"]),
    HumanExpert("hr_director", "Amanda Foster", ["human_resources", "change_management", "stakeholder_relations"], 12, 0.7, 0.3, ["labor", "community", "unions"]),
    HumanExpert("engineering_lead", "David Kim", ["engineering", "technology_assessment", "innovation"], 18, 0.8, 0.7, ["engineering", "innovation", "suppliers"])
]

print(f"Initialized {len(investment_projects)} illustrative projects and {len(human_experts)} expert profiles.")
print(f"Sum of individual project NPVs: ${sum(p['expected_npv'] for p in investment_projects.values()):.1f}M")

## 1. Explainable AI scenario analysis

The scenario database is a collection of randomly generated candidate portfolios. For each candidate, the code records its cost, market condition, market-adjusted scenario NPV, and NPV-to-cost ratio. If a candidate exceeds its generated scenario budget, projects are considered in descending base-NPV-per-cost order and retained while they fit. The recommendation stage filters this database against the decision budget and ranks the remaining candidates using the stated strategic-priority adjustments.

The stored historical success rates and implementation challenges below are synthetic teaching inputs. They are not observations from the 50 terminals mentioned in the illustrative scenario description.

In [ ]:
class ExplainableAI:
    """Sample and score candidate investment portfolios with transparent summaries."""

    def __init__(self, projects: Dict):
        self.projects = projects
        self.historical_data = self._generate_historical_data()
        self.scenario_database = self._create_scenario_database()

    def _generate_historical_data(self) -> Dict:
        """Return illustrative reference values, not observed terminal data."""
        return {
            "terminals_analyzed": 50,
            "implementations_studied": 200,
            "success_rates": {
                "autonomous_cranes": 0.78, "ai_yard_management": 0.85,
                "predictive_maintenance": 0.92, "digital_twin": 0.73,
                "smart_gates": 0.88
            },
            "average_roi": {
                "autonomous_cranes": 1.28, "ai_yard_management": 1.27,
                "predictive_maintenance": 1.20, "digital_twin": 1.40,
                "smart_gates": 1.25
            },
            "implementation_challenges": {
                "autonomous_cranes": ["labor resistance", "technical complexity", "regulatory approval"],
                "ai_yard_management": ["data quality", "integration complexity", "user adoption"],
                "predictive_maintenance": ["sensor deployment", "model accuracy", "maintenance culture"],
                "digital_twin": ["model accuracy", "computational resources", "expertise shortage"],
                "smart_gates": ["carrier adoption", "system integration", "privacy concerns"]
            }
        }

    def _create_scenario_database(self) -> List[Dict]:
        """Generate 10,000 random candidate scenarios for portfolio ranking."""
        scenarios = []
        project_ids = list(self.projects)
        market_multipliers = {"high_growth": 1.2, "stable": 1.0, "declining": 0.8}

        for _ in range(10000):
            scenario_budget = random.uniform(40, 120)
            market_condition = random.choice(list(market_multipliers))
            selected = random.sample(project_ids, random.randint(1, len(project_ids)))
            total_cost = sum(self.projects[p]["cost"] for p in selected)

            if total_cost > scenario_budget:
                selected.sort(
                    key=lambda p: self.projects[p]["expected_npv"] / self.projects[p]["cost"],
                    reverse=True
                )
                adjusted = []
                running_cost = 0.0
                for project_id in selected:
                    cost = self.projects[project_id]["cost"]
                    if running_cost + cost <= scenario_budget:
                        adjusted.append(project_id)
                        running_cost += cost
                selected = adjusted
                total_cost = running_cost

            base_npv = sum(self.projects[p]["expected_npv"] for p in selected)
            adjusted_npv = base_npv * market_multipliers[market_condition]
            scenarios.append({
                "budget": scenario_budget,
                "market_condition": market_condition,
                "selected_projects": selected,
                "total_cost": total_cost,
                "base_npv": base_npv,
                "total_npv": adjusted_npv,
                "roi": adjusted_npv / total_cost if total_cost else 0.0
            })
        return scenarios

    def generate_recommendation(self, budget: float, strategic_priorities: Dict[str, float]) -> AIRecommendation:
        """Choose the highest-scoring budget-feasible candidate in the sample."""
        viable = [s for s in self.scenario_database if s["total_cost"] <= budget]
        if not viable:
            raise ValueError("No sampled candidate portfolio satisfies the decision budget.")

        scored = []
        for scenario in viable:
            score = scenario["total_npv"]
            if "financial_return" in strategic_priorities:
                score *= 1 + strategic_priorities["financial_return"] * 0.5
            if "innovation" in strategic_priorities:
                bonus = sum(
                    0.2 for p in scenario["selected_projects"]
                    if p in {"digital_twin", "ai_yard_management"}
                )
                score *= 1 + bonus * strategic_priorities["innovation"]
            if "risk_averse" in strategic_priorities:
                risk = sum(self.projects[p]["risk_level"] for p in scenario["selected_projects"])
                score *= 1 - risk * strategic_priorities["risk_averse"] * 0.3
            scored.append((scenario, score))

        scored.sort(key=lambda item: item[1], reverse=True)
        best, best_score = scored[0]
        selected = best["selected_projects"]
        base_npv = best["base_npv"]
        reasoning = [
            f"Ranked {len(self.scenario_database)} sampled scenarios using the stated priority score.",
            f"Retained {len(viable)} sampled scenarios within the ${budget:.1f}M decision budget.",
            f"Applied priorities: {', '.join(strategic_priorities) if strategic_priorities else 'none specified'}.",
            f"Selected portfolio: {', '.join(self.projects[p]['name'] for p in selected) or 'no projects'}.",
            f"Base expected NPV: ${base_npv:.1f}M; base NPV-to-cost ratio: {base_npv / best['total_cost'] if best['total_cost'] else 0.0:.2f}."
        ]

        sensitivity = {
            "budget_variation": self._analyze_budget_sensitivity(budget, strategic_priorities),
            "market_conditions": self._analyze_market_sensitivity(best),
            "project_substitution": self._analyze_substitution_sensitivity(best)
        }
        alternatives = []
        for alternative, alternative_score in scored[1:4]:
            alternatives.append({
                "projects": [self.projects[p]["name"] for p in alternative["selected_projects"]],
                "npv": alternative["base_npv"],
                "cost": alternative["total_cost"],
                "difference_from_best_score": best_score - alternative_score,
                "key_advantage": self._get_alternative_advantage(best, alternative)
            })

        success_rates = [self.historical_data["success_rates"][p] for p in selected]
        historical_bonus = float(np.mean(success_rates)) * 0.2 if success_rates else 0.0
        matching = sum(set(s["selected_projects"]) == set(selected) for s in self.scenario_database)
        frequency_bonus = min(matching / 100, 0.1)
        confidence = min(0.7 + historical_bonus + frequency_bonus, 0.95)
        risks = [self.projects[p]["risk_level"] for p in selected]
        risk_assessment = {
            "implementation_risk": float(np.mean(risks)) if risks else 0.0,
            "financial_risk": max(0.0, (budget - best["total_cost"]) / budget * 0.3) if budget else 0.0,
            "market_risk": 0.15,
            "technology_risk": sum(0.1 for p in selected if p in {"autonomous_cranes", "digital_twin"})
        }
        risk_assessment["overall_risk"] = float(np.mean(list(risk_assessment.values())))

        return AIRecommendation(
            recommended_portfolio=[self.projects[p]["name"] for p in selected],
            expected_npv=base_npv,
            confidence_score=confidence,
            reasoning_path=reasoning,
            sensitivity_analysis=sensitivity,
            alternative_solutions=alternatives,
            risk_assessment=risk_assessment
        )

    def _analyze_budget_sensitivity(self, budget: float, priorities: Dict[str, float]) -> Dict[str, float]:
        results = {}
        for variation in [0.8, 0.9, 1.0, 1.1, 1.2]:
            varied_budget = budget * variation
            viable = [s for s in self.scenario_database if s["total_cost"] <= varied_budget]
            if viable:
                def score_scenario(scenario):
                    score = scenario["total_npv"]
                    if "financial_return" in priorities:
                        score *= 1 + priorities["financial_return"] * 0.5
                    if "innovation" in priorities:
                        count = sum(p in {"digital_twin", "ai_yard_management"} for p in scenario["selected_projects"])
                        score *= 1 + 0.2 * count * priorities["innovation"]
                    if "risk_averse" in priorities:
                        risk = sum(self.projects[p]["risk_level"] for p in scenario["selected_projects"])
                        score *= 1 - risk * priorities["risk_averse"] * 0.3
                    return score
                best = max(viable, key=score_scenario)
                results[f"{int(variation * 100)}%_budget"] = best["base_npv"]
        return results

    def _analyze_market_sensitivity(self, scenario: Dict) -> Dict[str, float]:
        multipliers = {"high_growth": 1.2, "stable": 1.0, "declining": 0.8}
        return {condition: scenario["base_npv"] * multiplier for condition, multiplier in multipliers.items()}

    def _analyze_substitution_sensitivity(self, scenario: Dict) -> Dict[str, float]:
        results = {}
        selected = scenario["selected_projects"]
        alternatives = [p for p in self.projects if p not in selected]
        if alternatives:
            best_alt = max(alternatives, key=lambda p: self.projects[p]["expected_npv"] / self.projects[p]["cost"])
            for project_id in selected:
                results[f"substitute_{project_id}"] = (
                    self.projects[best_alt]["expected_npv"] - self.projects[project_id]["expected_npv"]
                )
        return results

    def _get_alternative_advantage(self, best: Dict, alternative: Dict) -> str:
        if alternative["total_cost"] < best["total_cost"]:
            return f"Lower cost by ${best['total_cost'] - alternative['total_cost']:.1f}M"
        if len(alternative["selected_projects"]) < len(best["selected_projects"]):
            return "Fewer projects to implement"
        return "Different project mix and risk profile"

print("Explainable scenario-ranking system defined.")

## 2. Human-AI collaboration framework

Each expert profile generates structured priorities, concerns, risk factors, and implementation considerations. Priority aggregation uses a weighted average separately for each priority, so experts who did not express a particular priority do not dilute its value. The review and refinement rules are explicit, allowing the resulting portfolio and budget checks to be traced.

In [ ]:
class HumanAICollaboration:
    """Coordinate expert input, AI analysis, review, and implementation planning."""

    def __init__(self, experts: List[HumanExpert], ai_system: ExplainableAI):
        self.experts = {expert.id: expert for expert in experts}
        self.ai_system = ai_system
        self.collaboration_history = []
        self.trust_scores = {expert.id: 0.5 for expert in experts}
        self.learning_rate = 0.1

    def collect_human_inputs(self, decision_context: Dict) -> List[HumanInput]:
        return [
            HumanInput(
                expert_id=expert.id,
                strategic_priorities=self._generate_strategic_priorities(expert, decision_context),
                stakeholder_concerns=self._generate_stakeholder_concerns(expert),
                risk_factors=self._generate_risk_factors(expert),
                implementation_considerations=self._generate_implementation_considerations(expert),
                qualitative_factors=self._generate_qualitative_factors(expert)
            )
            for expert in self.experts.values()
        ]

    def _generate_strategic_priorities(self, expert: HumanExpert, context: Dict) -> Dict[str, float]:
        priorities = {}
        if "strategic_planning" in expert.expertise:
            priorities.update({"financial_return": 0.9, "market_positioning": 0.8})
        if "terminal_operations" in expert.expertise:
            priorities.update({"operational_excellence": 0.9, "safety": 0.8})
        if "human_resources" in expert.expertise:
            priorities.update({"labor_relations": 0.9, "social_responsibility": 0.8})
        if "engineering" in expert.expertise:
            priorities.update({"innovation": 0.9, "technical_feasibility": 0.8})
        return {key: min(1.0, value * random.uniform(0.8, 1.2)) for key, value in priorities.items()}

    def _generate_stakeholder_concerns(self, expert: HumanExpert) -> Dict[str, str]:
        concerns = {}
        concern_map = {
            "labor": {"job_displacement": "Automation may displace some roles", "skill_requirements": "Retraining may be needed"},
            "operations": {"operational_disruption": "Implementation may disrupt ongoing operations", "system_integration": "Integration with existing systems may be complex"},
            "community": {"local_employment": "Local employment may be affected", "community_relations": "Community engagement may be needed"},
            "investors": {"roi_timeline": "Investment payback timing is uncertain", "market_risk": "Market conditions may affect returns"}
        }
        for stakeholder in expert.stakeholder_focus:
            concerns.update(concern_map.get(stakeholder, {}))
        return concerns

    def _generate_risk_factors(self, expert: HumanExpert) -> List[str]:
        risks = ["Technology reliability", "Implementation delays", "Budget overruns", "Regulatory compliance", "Vendor dependency"]
        if expert.risk_tolerance < 0.5:
            risks.extend(["Organizational resistance", "Insufficient technical expertise", "Competitive response"])
        return risks

    def _generate_implementation_considerations(self, expert: HumanExpert) -> List[str]:
        considerations = ["Phased implementation", "Change management", "Training needs", "Vendor management", "Performance monitoring"]
        if "human_resources" in expert.expertise:
            considerations.extend(["Employee communication", "Union consultation", "Career transition support"])
        return considerations

    def _generate_qualitative_factors(self, expert: HumanExpert) -> Dict[str, float]:
        return {
            "strategic_alignment": random.uniform(0.7, 0.9),
            "organizational_readiness": random.uniform(0.6, 0.9),
            "market_timing": random.uniform(0.5, 0.9),
            "competitive_advantage": random.uniform(0.6, 0.9),
            "innovation_potential": random.uniform(0.7, 0.9)
        }

    def facilitate_collaborative_decision(self, budget: float, decision_context: Dict) -> CollaborativeDecision:
        print("Starting the human-AI collaborative decision process.")
        print("\nPhase 1: Problem framing")
        human_inputs = self.collect_human_inputs(decision_context)
        priorities = self._aggregate_strategic_priorities(human_inputs)
        print(f"Aggregated priority keys: {', '.join(priorities)}")

        print("\nPhase 2: AI scenario analysis")
        ai_rec = self.ai_system.generate_recommendation(budget, priorities)
        print(f"Sampled-candidate recommendation: {len(ai_rec.recommended_portfolio)} projects")
        print(f"Base expected NPV: ${ai_rec.expected_npv:.1f}M; illustrative confidence: {ai_rec.confidence_score:.1%}")

        print("\nPhase 3: Human review")
        feedback = self._collect_human_feedback(human_inputs, ai_rec)
        print(f"Review concerns recorded: {len(feedback['concerns'])}")

        print("\nPhase 4: Collaborative refinement")
        refined = self._develop_collaborative_solution(ai_rec, feedback, budget)
        print(f"Projects in refined portfolio: {len(refined)}")

        print("\nPhase 5: Implementation planning")
        plan = self._create_implementation_plan(refined, human_inputs)
        outcomes = self._calculate_expected_outcomes(refined)
        impacts = self._assess_stakeholder_impact(human_inputs)
        agreement = self._calculate_agreement_level(ai_rec, refined)
        confidence = (ai_rec.confidence_score + agreement) / 2
        self._update_trust_scores(agreement)

        decision = CollaborativeDecision(refined, outcomes, impacts, plan, confidence, agreement)
        self.collaboration_history.append(decision)
        return decision

    def _aggregate_strategic_priorities(self, human_inputs: List[HumanInput]) -> Dict[str, float]:
        weighted_totals = {}
        weight_totals = {}
        for human_input in human_inputs:
            weight = self.experts[human_input.expert_id].strategic_weight
            for priority, value in human_input.strategic_priorities.items():
                weighted_totals[priority] = weighted_totals.get(priority, 0.0) + value * weight
                weight_totals[priority] = weight_totals.get(priority, 0.0) + weight
        return {key: weighted_totals[key] / weight_totals[key] for key in weighted_totals}

    def _collect_human_feedback(self, human_inputs: List[HumanInput], ai_rec: AIRecommendation) -> Dict:
        feedback = {"concerns": [], "suggestions": [], "modifications": [], "additional_considerations": []}
        for human_input in human_inputs:
            expert = self.experts[human_input.expert_id]
            if "labor" in expert.stakeholder_focus and any("autonomous" in name.lower() for name in ai_rec.recommended_portfolio):
                feedback["concerns"].append(f"{expert.name}: labor displacement requires review")
            if "operations" in expert.stakeholder_focus and ai_rec.expected_npv < 30:
                feedback["concerns"].append(f"{expert.name}: operational benefits require review")
            if "change_management" in expert.expertise:
                feedback["suggestions"].append(f"{expert.name}: use phased implementation")
            if "financial_analysis" in expert.expertise:
                feedback["suggestions"].append(f"{expert.name}: review financing assumptions")
        return feedback

    def _develop_collaborative_solution(self, ai_rec: AIRecommendation, feedback: Dict, budget: float) -> List[str]:
        portfolio = ai_rec.recommended_portfolio.copy()
        project_by_name = {project["name"]: project for project in self.ai_system.projects.values()}
        labor_concern = any("labor" in concern.lower() for concern in feedback["concerns"])
        autonomous_name = self.ai_system.projects["autonomous_cranes"]["name"]
        maintenance_name = self.ai_system.projects["predictive_maintenance"]["name"]

        if labor_concern and autonomous_name in portfolio:
            candidate = [name for name in portfolio if name != autonomous_name]
            if maintenance_name not in candidate:
                candidate.append(maintenance_name)
            candidate_cost = sum(project_by_name[name]["cost"] for name in candidate)
            portfolio = candidate if candidate_cost <= budget else [name for name in portfolio if name != autonomous_name]

        assert sum(project_by_name[name]["cost"] for name in portfolio) <= budget + 1e-9
        return portfolio

    def _create_implementation_plan(self, portfolio: List[str], human_inputs: List[HumanInput]) -> Dict:
        phased = any("change_management" in self.experts[hi.expert_id].expertise for hi in human_inputs)
        milestones = []
        for index, project in enumerate(portfolio, start=1):
            milestones.extend([f"Month {index * 6}: {project} deployment review", f"Month {index * 12}: {project} full-operation review"])
        return {
            "approach": "phased implementation" if phased else "comprehensive implementation",
            "timeline_months": len(portfolio) * 12,
            "key_milestones": milestones,
            "risk_mitigation": ["Stakeholder communication", "Training", "Pilot testing", "Vendor monitoring"],
            "success_criteria": ["Budget adherence", "Operational performance", "Stakeholder feedback"],
            "stakeholder_engagement": ["Consult affected groups", "Review concerns at milestones"]
        }

    def _calculate_expected_outcomes(self, portfolio: List[str]) -> Dict[str, float]:
        project_by_name = {project["name"]: project for project in self.ai_system.projects.values()}
        total_cost = sum(project_by_name[name]["cost"] for name in portfolio)
        base_npv = sum(project_by_name[name]["expected_npv"] for name in portfolio)
        adjusted_npv = base_npv * 1.05
        return {
            "total_investment": total_cost,
            "base_expected_npv": base_npv,
            "expected_npv": adjusted_npv,
            "collaboration_uplift_assumption": 0.05,
            "roi": adjusted_npv / total_cost if total_cost else 0.0,
            "implementation_confidence": 0.85,
            "stakeholder_buy_in": 0.90
        }

    def _assess_stakeholder_impact(self, human_inputs: List[HumanInput]) -> Dict[str, str]:
        concerns = {}
        for human_input in human_inputs:
            for key, concern in human_input.stakeholder_concerns.items():
                concerns.setdefault(key, []).append(concern)
        impacts = {
            key: "Multiple concerns: " + "; ".join(items[:2]) if len(items) > 1 else items[0]
            for key, items in concerns.items()
        }
        impacts["organization"] = "Potential for improved competitive positioning and operational efficiency"
        impacts["customers"] = "Potential for improved service quality and reliability"
        return impacts

    def _calculate_agreement_level(self, ai_rec: AIRecommendation, refined_portfolio: List[str]) -> float:
        ai_set = set(ai_rec.recommended_portfolio)
        refined_set = set(refined_portfolio)
        union = ai_set | refined_set
        overlap = len(ai_set & refined_set) / len(union) if union else 0.5
        mean_trust = float(np.mean(list(self.trust_scores.values()))) if self.trust_scores else 0.5
        return overlap * 0.7 + mean_trust * 0.3

    def _update_trust_scores(self, agreement: float) -> None:
        for expert_id, current in self.trust_scores.items():
            updated = current + self.learning_rate * (agreement - current)
            self.trust_scores[expert_id] = max(0.1, min(0.95, updated))

print("Human-AI collaboration framework defined.")

## 3. Run the collaborative decision process

This cell creates the sampled scenario database, supplies the decision context, and runs the five collaboration phases. The budget is an input constraint. The portfolio's base NPV is reported separately from the assumed 5% collaboration uplift so the assumption remains visible.

In [ ]:
ai_system = ExplainableAI(investment_projects)
collaboration = HumanAICollaboration(human_experts, ai_system)

decision_context = {
    "organization_type": "port_authority",
    "current_automation_level": 0.3,
    "market_position": "competitive",
    "strategic_timeframe": "5_years",
    "risk_appetite": "moderate"
}
budget = 100.0
collaborative_decision = collaboration.facilitate_collaborative_decision(budget, decision_context)

print("\nCOLLABORATIVE DECISION SUMMARY")
print(f"Selected portfolio: {', '.join(collaborative_decision.selected_portfolio) or 'No projects'}")
print(f"Total investment: ${collaborative_decision.expected_outcomes['total_investment']:.1f}M")
print(f"Base expected NPV: ${collaborative_decision.expected_outcomes['base_expected_npv']:.1f}M")
print(f"NPV including illustrative collaboration uplift: ${collaborative_decision.expected_outcomes['expected_npv']:.1f}M")
print(f"NPV-to-investment ratio: {collaborative_decision.expected_outcomes['roi']:.2f}")
print(f"Illustrative confidence level: {collaborative_decision.confidence_level:.1%}")
print(f"Portfolio-overlap agreement measure: {collaborative_decision.human_ai_agreement:.1%}")
print(f"Implementation approach: {collaborative_decision.implementation_plan['approach']}")
print(f"Planning horizon: {collaborative_decision.implementation_plan['timeline_months']} months")

print("\nStakeholder review topics:")
for stakeholder, impact in collaborative_decision.stakeholder_impact.items():
    print(f"- {stakeholder.replace('_', ' ').title()}: {impact}")

print("\nTrust scores after this decision:")
for expert_id, trust in collaboration.trust_scores.items():
    print(f"- {collaboration.experts[expert_id].name}: {trust:.1%}")

project_by_name = {project['name']: project for project in investment_projects.values()}
assert sum(project_by_name[name]['cost'] for name in collaborative_decision.selected_portfolio) <= budget + 1e-9
assert np.isclose(
    collaborative_decision.expected_outcomes['total_investment'],
    sum(project_by_name[name]['cost'] for name in collaborative_decision.selected_portfolio)
)
assert np.isclose(
    collaborative_decision.expected_outcomes['base_expected_npv'],
    sum(project_by_name[name]['expected_npv'] for name in collaborative_decision.selected_portfolio)
)
print("\nBudget and portfolio-total checks passed.")

## 4. Illustrative comparison with individual approaches

The pure-human portfolio below is a rule-based teaching benchmark: it considers three specified projects in a fixed order and applies an assumed 10% NPV reduction. Stakeholder buy-in, implementation success, and confidence values are also assigned assumptions. The collaborative result includes the explicit 5% uplift assumption. These values are not measured outcomes, and the comparison does not prove that collaboration improves financial value.

In [ ]:
pure_ai_priorities = {"financial_return": 0.8, "innovation": 0.6}
pure_ai_recommendation = ai_system.generate_recommendation(budget, pure_ai_priorities)


def simulate_pure_human_decision(budget: float, projects: Dict) -> Dict:
    """Create a fixed-order, conservative illustrative benchmark portfolio."""
    order = ["predictive_maintenance", "smart_gates", "ai_yard_management"]
    selected_ids = []
    total_cost = 0.0
    for project_id in order:
        cost = projects[project_id]["cost"]
        if total_cost + cost <= budget:
            selected_ids.append(project_id)
            total_cost += cost
    base_npv = sum(projects[p]["expected_npv"] for p in selected_ids)
    return {
        "portfolio": [projects[p]["name"] for p in selected_ids],
        "cost": total_cost,
        "npv": base_npv * 0.9,
        "confidence": 0.65,
        "implementation_success": 0.85,
        "stakeholder_buy_in": 0.95
    }


pure_human_decision = simulate_pure_human_decision(budget, investment_projects)
pure_ai_cost = sum(project_by_name[name]["cost"] for name in pure_ai_recommendation.recommended_portfolio)

comparison_data = {
    "Approach": ["Pure AI sample", "Pure human benchmark", "Human-AI collaboration"],
    "NPV ($M)": [
        pure_ai_recommendation.expected_npv,
        pure_human_decision["npv"],
        collaborative_decision.expected_outcomes["expected_npv"]
    ],
    "Investment ($M)": [
        pure_ai_cost,
        pure_human_decision["cost"],
        collaborative_decision.expected_outcomes["total_investment"]
    ],
    "NPV / Investment": [
        pure_ai_recommendation.expected_npv / pure_ai_cost if pure_ai_cost else 0.0,
        pure_human_decision["npv"] / pure_human_decision["cost"] if pure_human_decision["cost"] else 0.0,
        collaborative_decision.expected_outcomes["roi"]
    ],
    "Illustrative confidence": [
        pure_ai_recommendation.confidence_score,
        pure_human_decision["confidence"],
        collaborative_decision.confidence_level
    ],
    "Assumed stakeholder buy-in": [
        0.70,
        pure_human_decision["stakeholder_buy_in"],
        collaborative_decision.expected_outcomes["stakeholder_buy_in"]
    ],
    "Assumed implementation success": [
        0.75,
        pure_human_decision["implementation_success"],
        0.95
    ]
}
comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.round(3).to_string(index=False))

best_individual_npv = max(pure_ai_recommendation.expected_npv, pure_human_decision["npv"])
collaborative_npv = collaborative_decision.expected_outcomes["expected_npv"]
npv_difference_percent = (collaborative_npv - best_individual_npv) / best_individual_npv * 100 if best_individual_npv else 0.0
print("\nIllustrative comparison against the higher-NPV individual benchmark:")
print(f"NPV difference: {npv_difference_percent:.1f}%")
print("Interpret this difference in light of the benchmark assumptions and sampled AI scenarios.")

## 5. Visual summaries

The panels use separate axes for financial values, percentage-style scores, trust, portfolio membership, and NPV comparisons. The trust trajectory is hypothetical. Portfolio colors are labeled by a color bar, and the last panel compares the three benchmark NPVs rather than assigning unsupported shares of collaborative value to humans and AI.

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

approaches = comparison_df["Approach"]
x = np.arange(len(approaches))
width = 0.36
ax1.bar(x - width / 2, comparison_df["NPV ($M)"], width, label="NPV ($M)", color="steelblue")
ax1.bar(x + width / 2, comparison_df["Investment ($M)"], width, label="Investment ($M)", color="darkorange")
ax1.set_xticks(x)
ax1.set_xticklabels(approaches, rotation=20, ha="right")
ax1.set_ylabel("Value ($M)")
ax1.set_title("Illustrative Financial Comparison")
ax1.legend()
ax1.grid(axis="y", alpha=0.3)

time_periods = ["Initial", "After cycle 1", "After cycle 3", "After cycle 5"]
trust_evolution = {
    "CEO": [0.50, 0.65, 0.78, 0.85],
    "Operations director": [0.50, 0.70, 0.82, 0.90],
    "HR director": [0.50, 0.60, 0.75, 0.88],
    "Engineering lead": [0.50, 0.75, 0.85, 0.92]
}
for expert_name, scores in trust_evolution.items():
    ax2.plot(time_periods, scores, marker="o", linewidth=2, label=expert_name)
ax2.set_xlabel("Illustrative collaboration stage")
ax2.set_ylabel("Trust score (0–1)")
ax2.set_title("Hypothetical Trust Trajectory")
ax2.set_ylim(0.4, 1.0)
ax2.legend()
ax2.grid(alpha=0.3)

portfolio_data = {
    "Pure AI sample": pure_ai_recommendation.recommended_portfolio,
    "Pure human benchmark": pure_human_decision["portfolio"],
    "Human-AI collaboration": collaborative_decision.selected_portfolio
}
all_projects = list(dict.fromkeys(name for portfolio in portfolio_data.values() for name in portfolio))
portfolio_matrix = np.array([
    [int(project in portfolio) for project in all_projects]
    for portfolio in portfolio_data.values()
])
image = ax3.imshow(portfolio_matrix, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax3.set_xticks(range(len(all_projects)))
ax3.set_xticklabels(all_projects, rotation=40, ha="right")
ax3.set_yticks(range(len(portfolio_data)))
ax3.set_yticklabels(list(portfolio_data))
ax3.set_title("Portfolio Membership (0 = Not Selected, 1 = Selected)")
fig.colorbar(image, ax=ax3, ticks=[0, 1], label="Selection indicator")

npv_values = comparison_df["NPV ($M)"].to_numpy()
bars = ax4.bar(comparison_df["Approach"], npv_values, color=["steelblue", "darkorange", "seagreen"])
ax4.set_ylabel("NPV ($M)")
ax4.set_title("NPV Under the Stated Benchmark Assumptions")
ax4.tick_params(axis="x", labelrotation=20)
ax4.grid(axis="y", alpha=0.3)
for bar in bars:
    ax4.annotate(
        f"${bar.get_height():.1f}M",
        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 4), textcoords="offset points", ha="center", va="bottom"
    )

fig.suptitle("Human-AI Investment Decision: Illustrative Summaries", fontsize=15)
fig.tight_layout()
plt.show()

## 6. Illustrative learning and improvement analysis

The following function specifies an increasing sequence of hypothetical metrics to demonstrate how a learning analysis can be organized. It does not use observed decision outcomes, and its values are not connected to the trust update in the collaboration object. The computed percentage changes describe only this assumed sequence.

In [ ]:
def simulate_learning_cycles(num_cycles: int = 5) -> List[PartnershipMetrics]:
    """Create a hypothetical metric trajectory for teaching purposes."""
    if num_cycles < 1:
        raise ValueError("num_cycles must be at least 1")
    history = []
    for cycle in range(num_cycles):
        history.append(PartnershipMetrics(
            decision_quality_score=min(0.75 + cycle * 0.03, 0.95),
            stakeholder_satisfaction=min(0.80 + cycle * 0.02, 0.95),
            implementation_success=min(0.85 + cycle * 0.02, 0.98),
            trust_level=min(0.50 + cycle * 0.10, 0.90),
            learning_rate=0.10 + cycle * 0.02
        ))
    return history


learning_metrics = simulate_learning_cycles(5)
print("Hypothetical partnership learning trajectory:")
for cycle, metrics in enumerate(learning_metrics, start=1):
    print(f"\nCycle {cycle}:")
    print(f"  Decision quality: {metrics.decision_quality_score:.1%}")
    print(f"  Stakeholder satisfaction: {metrics.stakeholder_satisfaction:.1%}")
    print(f"  Implementation success: {metrics.implementation_success:.1%}")
    print(f"  Trust level: {metrics.trust_level:.1%}")
    print(f"  Learning rate: {metrics.learning_rate:.3f}")

initial_metrics = learning_metrics[0]
final_metrics = learning_metrics[-1]
improvements = {
    "Decision quality": (final_metrics.decision_quality_score - initial_metrics.decision_quality_score) / initial_metrics.decision_quality_score * 100,
    "Stakeholder satisfaction": (final_metrics.stakeholder_satisfaction - initial_metrics.stakeholder_satisfaction) / initial_metrics.stakeholder_satisfaction * 100,
    "Implementation success": (final_metrics.implementation_success - initial_metrics.implementation_success) / initial_metrics.implementation_success * 100,
    "Trust level": (final_metrics.trust_level - initial_metrics.trust_level) / initial_metrics.trust_level * 100
}
print("\nPercentage changes within this hypothetical sequence:")
for metric, change in improvements.items():
    print(f"- {metric}: {change:.1f}%")

success_factors = [
    "Complementary expertise between human participants and analytical systems",
    "Transparent reasoning that supports review",
    "A structured decision process with clear roles",
    "Trust measurement informed by outcomes",
    "Feedback loops that can inform later decisions",
    "Consideration of quantitative and qualitative factors",
    "Stakeholder-focused implementation planning"
]
recommendations = [
    "Define collaboration protocols and responsibilities",
    "Make assumptions and analytical reasoning reviewable",
    "Monitor trust and decision measures without treating them as proof of performance",
    "Record feedback and compare forecasts with observed outcomes",
    "Prepare experts to interpret model outputs and limitations",
    "Maintain clear communication with affected stakeholders",
    "Document decisions and implementation lessons"
]
print("\nPotential factors to examine in a real evaluation:")
for number, factor in enumerate(success_factors, start=1):
    print(f"{number}. {factor}")
print("\nPractical recommendations for a real decision process:")
for number, recommendation in enumerate(recommendations, start=1):
    print(f"{number}. {recommendation}")

print("\nAnalysis complete. All numerical comparisons and learning curves are illustrative.")